In [ ]:
# ============================================================
# Text preprocessing, NER, POS tagging et Word2Vec
# ============================================================

# Installation des bibliothèques nécessaires (à exécuter une fois dans Colab)
# --------------------------------------------------------------------------
# !pip install -q nltk spacy gensim matplotlib scikit-learn
# !python -m spacy download en_core_web_sm

# ============================================================
# 1. Imports et préparation
# ============================================================
import string

import nltk
import spacy
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

# Téléchargement des ressources nécessaires pour nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")

# Chargement du modèle spaCy
nlp = spacy.load("en_core_web_sm")

# ============================================================
# 2. Jeu de données brut (raw data)
# ============================================================
data = {
    "Review": [
        "At McDonald's the food was ok and the service was bad.",
        "I would not recommend this Japanese restaurant to anyone.",
        "I loved this restaurant when I traveled to Thailand last summer.",
        "The menu of Loving has a wide variety of options.",
        "The staff was friendly and helpful at Google's employees restaurant.",
        "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
        "I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
        "The sushi at Sushi Express is always fresh and flavorful.",
        "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
        "The dessert selection at Sweet Treats is to die for!"
    ]
}

raw_reviews = data["Review"]

print("=== Aperçu des données brutes ===")
for i, review in enumerate(raw_reviews, start=1):
    print(f"Ligne {i} :", review)


# ============================================================
# EXERCICE 1
# ============================================================

# ------------------------------------------------------------
# 1. Fonction preprocess_text()
#    - mettre en minuscules
#    - tokeniser
#    - supprimer la ponctuation
#    - supprimer les stopwords
#    - appliquer un lemmatizer
#    - renvoyer la phrase prétraitée (chaine de caractères)
# ------------------------------------------------------------

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))
punctuation_set = set(string.punctuation)


def preprocess_text(text):
    """
    Prétraitement du texte :
    - mise en minuscules
    - tokenisation
    - suppression de la ponctuation
    - suppression des stopwords
    - lemmatisation
    Retourne une chaine de caractères nettoyée.
    """
    # Mise en minuscules
    text_lower = text.lower()

    # Tokenisation
    tokens = word_tokenize(text_lower)

    # Suppression de la ponctuation et des tokens non alphabétiques
    tokens = [
        token
        for token in tokens
        if token.isalpha() and token not in punctuation_set
    ]

    # Suppression des stopwords
    tokens = [token for token in tokens if token not in stop_words]

    # Lemmatisation
    lemmas = [lemmatizer.lemmatize(token) for token in tokens]

    # Reconstruction en chaine de caractères
    cleaned_text = " ".join(lemmas)
    return cleaned_text


# Application de la fonction sur le jeu de données
preprocessed_reviews = [preprocess_text(review) for review in raw_reviews]

print("\n=== Vérification de preprocess_text() sur quelques lignes ===")
for i, (raw, clean) in enumerate(zip(raw_reviews, preprocessed_reviews), start=1):
    print(f"\nLigne {i} - Texte brut :")
    print(raw)
    print("Texte prétraité :")
    print(clean)


# ------------------------------------------------------------
# 2. Création d un nouveau jeu de données avec le texte nettoyé
#    (nous conservons :
#     - raw_reviews : données brutes
#     - preprocessed_reviews : données nettoyées)
# ------------------------------------------------------------

data_preprocessed = {
    "Review_raw": raw_reviews,
    "Review_clean": preprocessed_reviews
}

print("\n=== Aperçu du nouveau jeu de données (brut et nettoyé) ===")
for i in range(len(raw_reviews)):
    print(f"\nLigne {i + 1} :")
    print("Brut   :", data_preprocessed["Review_raw"][i])
    print("Nettoyé:", data_preprocessed["Review_clean"][i])


# ------------------------------------------------------------
# 3. Fonction perform_ner()
#    - reçoit un texte
#    - applique spaCy pour faire la reconnaissance d entités nommées
#    - retourne une liste de tuples (entité, label)
# ------------------------------------------------------------

def perform_ner(text):
    """
    Applique la reconnaissance d entités nommées sur un texte.
    Utilise spaCy en_core_web_sm.
    Retourne une liste de tuples (texte_entité, label_entité).
    """
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities


# Test de perform_ner sur quelques exemples
print("\n=== Test de perform_ner() sur les données brutes ===")
for i, review in enumerate(raw_reviews[:3], start=1):
    ents = perform_ner(review)
    print(f"\nLigne {i} : {review}")
    print("Entités nommées :", ents)

print("\n=== Test de perform_ner() sur les données prétraitées ===")
for i, review in enumerate(preprocessed_reviews[:3], start=1):
    ents = perform_ner(review)
    print(f"\nLigne prétraitée {i} : {review}")
    print("Entités nommées :", ents)


# ------------------------------------------------------------
# 4. Fonction perform_pos_tagging()
#    - reçoit un texte
#    - fait la tokenisation
#    - applique nltk.pos_tag
#    - retourne la liste des (token, tag)
# ------------------------------------------------------------

def perform_pos_tagging(text):
    """
    Applique le POS tagging sur un texte donné.
    Utilise la fonction pos_tag de nltk.
    Retourne une liste de tuples (token, tag).
    """
    tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    return pos_tags


# Test de perform_pos_tagging sur brut et prétraité
print("\n=== Test de perform_pos_tagging() sur les données brutes ===")
for i, review in enumerate(raw_reviews[:3], start=1):
    tags = perform_pos_tagging(review)
    print(f"\nLigne {i} : {review}")
    print("POS tags :", tags)

print("\n=== Test de perform_pos_tagging() sur les données prétraitées ===")
for i, review in enumerate(preprocessed_reviews[:3], start=1):
    tags = perform_pos_tagging(review)
    print(f"\nLigne prétraitée {i} : {review}")
    print("POS tags :", tags)

# Pour explorer la signification des tags, si besoin :
# nltk.download("tagsets")
# nltk.help.upenn_tagset("NN")


# ============================================================
# EXERCICE 2 : Word embeddings et visualisation
# ============================================================

# ------------------------------------------------------------
# 1. Création des embeddings Word2Vec
#    - Utiliser le jeu de données prétraité
#    - Tokeniser en liste de mots pour chaque phrase
# ------------------------------------------------------------

# Tokenisation des phrases prétraitées pour Word2Vec
tokenized_clean_reviews = [sentence.split() for sentence in preprocessed_reviews]

# Création du modèle Word2Vec
# vector_size : dimension des vecteurs
# window      : taille de la fenêtre de contexte
# min_count   : fréquence minimale pour inclure un mot
# sg          : 0 pour modèle CBOW, 1 pour modèle Skip-gram
word2vec_model = Word2Vec(
    sentences=tokenized_clean_reviews,
    vector_size=50,
    window=5,
    min_count=1,
    workers=4,
    sg=0
)

print("\n=== Informations sur le modèle Word2Vec ===")
print("Nombre de mots dans le vocabulaire :", len(word2vec_model.wv))
print("Dimension des vecteurs :", word2vec_model.vector_size)
print(
    "\nInterprétation : chaque mot est représenté par un vecteur de taille "
    f"{word2vec_model.vector_size}. Cela signifie que chaque mot est projeté "
    "dans un espace vectoriel de cette dimension, où la proximité géométrique "
    "reflète une certaine proximité sémantique ou contextuelle."
)


# ------------------------------------------------------------
# 2. Fonction plot_word_embeddings()
#    - reçoit un modèle Word2Vec
#    - projette les vecteurs en deux dimensions (PCA)
#    - affiche un nuage de points (scatter plot)
#    - ajoute les mots comme étiquettes avec annotate()
# ------------------------------------------------------------

def plot_word_embeddings(w2v_model, max_words=30):
    """
    Visualise les embeddings Word2Vec en deux dimensions.
    Utilise une réduction de dimension par PCA.
    max_words limite le nombre de mots affichés pour garder
    un graphique lisible.
    """
    words = list(w2v_model.wv.index_to_key)

    # Limite du nombre de mots pour la visualisation
    words = words[:max_words]

    # Récupération des vecteurs
    vectors = [w2v_model.wv[word] for word in words]

    # Réduction de dimension à deux dimensions avec PCA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(vectors)

    x_coords = coords[:, 0]
    y_coords = coords[:, 1]

    plt.figure(figsize=(10, 8))
    plt.scatter(x_coords, y_coords)

    for i, word in enumerate(words):
        plt.annotate(word, (x_coords[i], y_coords[i]))

    plt.title("Projection en deux dimensions des embeddings Word2Vec")
    plt.xlabel("Composante principale un")
    plt.ylabel("Composante principale deux")
    plt.grid(True)
    plt.show()


# Appel de la fonction de visualisation
plot_word_embeddings(word2vec_model, max_words=30)

print(
    "\nAnalyse possible :\n"
    "- Les mots qui apparaissent dans des contextes similaires "
    "devraient être relativement proches sur le graphique.\n"
    "- Si certains mots attendus ne sont pas très bien regroupés, "
    "cela peut être dû à la petite taille du corpus, aux paramètres "
    "du modèle Word2Vec, ou au prétraitement (suppression de certains mots, "
    "lemmatisation, fenêtre de contexte, et ainsi de suite).\n"
    "- En modifiant la fenêtre, la dimension des vecteurs, ou la stratégie "
    "du modèle (CBOW versus Skip-gram), vous pouvez obtenir des "
    "représentations légèrement différentes."
)

# ------------------------------------------------------------
# 3. Pistes d amélioration
# ------------------------------------------------------------
print(
    "Pistes pour aller plus loin :\n"
    "1. Tester des variantes de prétraitement (conserver certains adjectifs,\n"
    "   retirer moins de mots, traiter les noms propres différemment).\n"
    "2. Ajuster les paramètres du modèle Word2Vec (dimension, fenêtre,\n"
    "   choix CBOW ou Skip-gram, min_count, nombre d itérations).\n"
    "3. Utiliser d autres techniques de visualisation (t SNE, UMAP) pour\n"
    "   mieux séparer visuellement des groupes de mots similaires."
)
